# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 21_surrogate_model_training  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Properties script
**Goal:** To train a surrogate model on a CSV dataset resulting from structural analyses done in grasshopper, this surrogate model should be able to accurately tell if a beam in the structure is structurally failing or not   
**Inputs:**
*   CSV with structural properties, grasshopper

**Outputs:**
*   A surrogate model????

# IMPORTING DATA

In [ ]:
import pandas as pd
import numpy as np
import joblib
import config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# 1. CONFIGURATIE (Vul hier later je definitieve kolommen in)
# ==========================================
CSV_FILE = '14_dataset_results.csv'

# Stel hier in welke kolommen je als input (X) en target (y) wilt gebruiken.
# Voorbeeld voor je latere dataset: INPUT_COLS = ['v4_u', 'v4_v', 'v5_shift_x', ...]
# Omdat deze nu nog ontbreken in de colibri file, doen we alsof dit de inputs zijn:
INPUT_COLS = ['out:beams_length', 'out:mass_structure']

# TARGETS (Waar de AI op gaat sturen)
# Bijvoorbeeld: totale CO2, doorbuiging en constructieve efficiëntie
TARGET_COLS = ['out:max_displacement', 'out:total_utilization']

print("1. Dataset inladen...")
print("...")
print("...")
print("...")
print("...")
print("...")
df = pd.read_csv(config.RAW_DATA_PATH + CSV_FILE)
print(f"✅ Dataset '{CSV_FILE}' ingeladen.")

print(f"\nTotaal aantal samples in dataset CSV: {len(df)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(df.head(5))

1. Dataset inladen...
...
...
...
...
...
✅ Dataset '14_dataset_results.csv' ingeladen.

Totaal aantal samples in dataset CSV: 1001

Voorbeeld output (eerste 5 regels van sample 0):
   in:num_sample  out:beams_length  out:mass_structure  out:max_displacement  \
0              0        100.117906           62.535678              1.459735   
1              1         91.499631           49.554857              1.027687   
2              2         93.896324           52.854899              0.850376   
3              3         91.499518           51.139870              0.786957   
4              4         93.254981           51.532886              0.913399   

   out:total_utilization  img  
0              42.895888  NaN  
1              22.355890  NaN  
2              24.732524  NaN  
3              26.571157  NaN  
4              25.870203  NaN  


## Import Edge Index

In [ ]:
import pandas as pd
import numpy as np
from config import RAW_DATA_PATH

# Inladen van de dataset
df_edge = pd.read_csv(RAW_DATA_PATH + 'dataset_2x2_10_edges.csv')

# Filteren op sample_id 0 om de basis-topologie te extraheren
# We gaan ervan uit dat de eerste kolom de sample_id is (bijv. kolom 0)
sample_col = "sample_id"
df_topology = df_edge[df_edge[sample_col] == 0].copy()

print(f"Topologie geëxtraheerd uit {sample_col} == 0.")
print(f"Aantal unieke verbindingen in de basisstructuur: {len(df_topology)}")

# De kolommen V0 en V1 omzetten naar de 2xE edge_index
edge_index_np = df_topology[['V1', 'V2']].values.T

# Omzetten naar lijst voor je workflow
edge_index = edge_index_np.tolist()

print("\nGegenereerde edge_index (eerste 5 verbindingen):")
print(f"V0 (bron): {edge_index[0][:5]}")
print(f"V1 (doel): {edge_index[1][:5]}")

Topologie geëxtraheerd uit sample_id == 0.
Aantal unieke verbindingen in de basisstructuur: 32

Gegenereerde edge_index (eerste 5 verbindingen):
V0 (bron): [0, 0, 1, 1, 2]
V1 (doel): [1, 3, 2, 4, 5]


# MODEL TRAINING

In [4]:
# Verwijder lege/irrelevante kolommen (bijv. de lege 'img' kolom van Colibri)
if 'img' in df.columns:
    df = df.drop(columns=['img'])

# Drop eventuele rijen met lege waarden (NaN)
df = df.dropna()

print(f"Dataset ingeladen met {len(df)} samples.")

# ==========================================
# 2. X EN Y SPLITSEN (Features en Targets)
# ==========================================
X = df[INPUT_COLS]
y = df[TARGET_COLS]

# Splits de data: 80% trainen, 20% testen (evalueren hoe goed de AI het doet)
# random_state zorgt ervoor dat je elke keer dezelfde splitsing krijgt (reproduceerbaarheid!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Data gesplitst: {len(X_train)} trainingsamples en {len(X_test)} testsamples.")

# ==========================================
# 3. NORMALISATIE (Min-Max Scaling naar domein 0 - 1)
# ==========================================
print("3. Data normaliseren...")

# Maak twee aparte 'meetlatten' aan: één voor de inputs, één voor de targets
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# BELANGRIJK: De scalers "leren" (fit) de minimum/maximum waarden
# UITSLUITEND van de trainingsdata, om spieken (data leakage) te voorkomen.
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)

# Transformeer de test-data met de meetlat die zojuist is gemaakt
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

# Converteer terug naar DataFrames voor de overzichtelijkheid
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=INPUT_COLS)
y_train_scaled_df = pd.DataFrame(y_train_scaled, columns=TARGET_COLS)

print("\nVoorbeeld van de genormaliseerde Training Data (X):")
print(X_train_scaled_df.head(3))

# ==========================================
# 4. EXPORTEREN VOOR HET AI MODEL
# ==========================================
# We slaan de meetlatten op als .pkl bestanden.
# Deze heb je later nodig om de voorspellingen van je AI weer om te rekenen naar echte millimeters en kilo's!
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')

print("\n✅ Scalers succesvol opgeslagen als 'scaler_X.pkl' en 'scaler_y.pkl'")

# Nu is je data klaar om een neuraal netwerk (bijv met PyTorch/Keras) of een
# Random Forest model in te gooien!
# De variabelen die je nu aan je model voedt zijn: X_train_scaled en y_train_scaled

Dataset ingeladen met 1001 samples.
Data gesplitst: 800 trainingsamples en 201 testsamples.
3. Data normaliseren...

Voorbeeld van de genormaliseerde Training Data (X):
   out:beams_length  out:mass_structure
0          0.588958            0.601601
1          0.512405            0.202104
2          0.473171            0.379873

✅ Scalers succesvol opgeslagen als 'scaler_X.pkl' en 'scaler_y.pkl'


# MODEL TESTING

In [5]:
# ==========================================
# 2. HET NEURALE NETWERK BOUWEN (Surrogate Model)
# ==========================================
print("\n⚙️ Neuraal Netwerk aan het trainen...")

# Een Multi-Layer Perceptron (MLP).
# hidden_layer_sizes=(64, 64) betekent: 2 verborgen lagen met elk 64 'neuronen'.
# max_iter=1000 betekent: Het netwerk mag 1000 keer proberen de foutmarge te verkleinen.
surrogate_model = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

# Geef de genormaliseerde trainingsdata aan het model om te 'leren'
surrogate_model.fit(X_train_scaled, y_train_scaled)
print("✅ Training voltooid!")

# ==========================================
# 3. HET MODEL TESTEN (Evaluatie op ongeziene data)
# ==========================================
# Laat het model nu gokken (voorspellen) op de X_test data, die hij nog nooit gezien heeft!
y_pred_scaled = surrogate_model.predict(X_test_scaled)

# Bereken hoe goed de gok is vergeleken met de échte antwoorden (y_test_scaled)
r2 = r2_score(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)

print("\n📊 --- RESULTATEN VAN HET MODEL ---")
print(f"R2 Score (Determinatiecoëfficiënt): {r2:.4f} (1.0 is perfect)")
print(f"Mean Squared Error (Foutmarge):   {mse:.4f} (0.0 is perfect)")

# ==========================================
# 4. TERUGREKENEN NAAR DE ECHTE WERELD (Inferentie-check)
# ==========================================
# Hier gebruiken we de scaler om de genormaliseerde (0-1) voorspellingen
# weer terug te transformeren naar fysieke millimeters en percentages!
y_pred_fysiek = scaler_y.inverse_transform(y_pred_scaled)
y_test_fysiek = scaler_y.inverse_transform(y_test_scaled) # De daadwerkelijke test antwoorden

# Laat een willekeurige constructie-iteratie zien (bijv. iteratie nummer 5 uit de test-set)
voorbeeld_idx = 5
print(f"\n🔍 --- VOORBEELD ITERATIE ---")
print(f"Werkelijke {TARGET_COLS[0]}: \t{y_test_fysiek[voorbeeld_idx][0]:.3f}")
print(f"AI Voorspelt {TARGET_COLS[0]}: \t{y_pred_fysiek[voorbeeld_idx][0]:.3f}")
print("-" * 30)
print(f"Werkelijke {TARGET_COLS[1]}: \t{y_test_fysiek[voorbeeld_idx][1]:.3f}")
print(f"AI Voorspelt {TARGET_COLS[1]}: \t{y_pred_fysiek[voorbeeld_idx][1]:.3f}")

# Exporteer het getrainde netwerk zodat we het later in de optimalisatie-loop kunnen inladen!
joblib.dump(surrogate_model, 'surrogate_model.pkl')
print("\n💾 Model succesvol opgeslagen als 'surrogate_model.pkl'")


⚙️ Neuraal Netwerk aan het trainen...
✅ Training voltooid!

📊 --- RESULTATEN VAN HET MODEL ---
R2 Score (Determinatiecoëfficiënt): 0.8465 (1.0 is perfect)
Mean Squared Error (Foutmarge):   0.0034 (0.0 is perfect)

🔍 --- VOORBEELD ITERATIE ---
Werkelijke out:max_displacement: 	1.482
AI Voorspelt out:max_displacement: 	0.730
------------------------------
Werkelijke out:total_utilization: 	19.532
AI Voorspelt out:total_utilization: 	20.368

💾 Model succesvol opgeslagen als 'surrogate_model.pkl'
